# Install dependencies

In [ ]:
%pip install -q -U pip
%pip install -q langchain==1.0.2
%pip install -q langchain-community==0.4.1
%pip install -q langchain-core==1.0.1
%pip install -q langchain-huggingface==1.0.0
%pip install -q langchain-text-splitters==1.0.0
%pip install -q langgraph==1.0.1
%pip install -q langgraph==1.0.1
%pip install -q transformers==4.57.1

# LLM instantiation

In [ ]:
# LLM
import os
import torch
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
)

# Create a text generation pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, temperature=0.1, return_full_text=False)

llm = HuggingFacePipeline(pipeline=pipe)

# Langgraph Workflow Definition

This is the core of the agent's logic, defining how different components (nodes) interact and transition based on the current state and decisions. Langgraph allows for complex, stateful agent behaviors.


# Graph State

The GraphState defines the structure of information that flows through the Langgraph workflow.

In [ ]:
from typing_extensions import TypedDict
from typing import List
# from langchain.schema import Document
from langchain_core.messages import BaseMessage

############# State #################

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        web_search: whether to add internet search
    """

    question: str
    generation: str
    config: dict
    chat_history: List[BaseMessage]

# Session UUID loader

This function loads a random UUID. UUID is important for the graph to load a chat thread's previous memory

In [ ]:
import uuid
def load_session_id():
    session_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": session_id}}
    return session_id, config

In [ ]:
def load_chat_history(config):
    print("Session ID input, loading previous chat...")
    chat_history = list(app.get_state_history(config))
    # print("config: ", config)
    
    return chat_history

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage # Import necessary message types

def retrieve_chat_history(state: GraphState, config: RunnableConfig) -> GraphState:
    print("---Loading chat history---")

    current_question = state.get("question", "")
    chat_history = state.get("chat_history", [])
    thread_id = config['metadata']['thread_id']
    config = {"configurable": {"thread_id": thread_id}}
    return {"chat_history": chat_history, "question": current_question, "thread_id": thread_id}

# Chat output
This chain handles general conversation

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat_prompt = PromptTemplate(
    template="""<|begin_of_text|><|start_header_id|>system<|end_header_id|> You are a friendly chatbot that interacts with users. 
    Answer any question with the best of your abilities with the available context. Answer truthfully and friendly 
    Use three sentences maximum and keep the answer concise <|eot_id|><|start_header_id|>user<|end_header_id|>
    Chat History: {chat_history}
    Question: {question} 
    Answer: <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
    input_variables=["question", "chat_history"],
)

# Chain
chat_chain = chat_prompt | llm | StrOutputParser()

In [ ]:
def chat(state, config):
    """
    Normal chatbot without any grounding except for chat history.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---CHATTING---")
    question = state.get("question", "")
    
    chat_history = state.get("chat_history", []) 

    # Chat generation
    chat_response = chat_chain.invoke({"question": question, "chat_history": chat_history})

    build_history = {'question': question, 'generation': AIMessage(content=chat_response)}

    chat_history.insert(0, build_history)
    
    return {"question": question, "generation": chat_response, "chat_history": chat_history}

# Workflow Construction

This section defines the Langgraph workflow, connecting the nodes with edges and conditional edges to create the agent's decision-making flow.

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("chat_history", retrieve_chat_history)
workflow.add_node("chat", chat)


In [ ]:
# Build graph
workflow.add_edge(START, "chat_history")
workflow.add_edge("chat_history", "chat")

workflow.add_edge("chat", END)

checkpointer = InMemorySaver()

app = workflow.compile(checkpointer)

In [ ]:
from IPython.display import Image, display
# app = workflow.compile(checkpointer=checkpointer)
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("error")
    # This requires some extra dependencies and is optional
    pass

# UUID creation
This code calls the load_session_id() function run on the previous steps to generate a randomly generated UUID which allows the chat history to be saved and loaded.

In [ ]:
session_id, config = load_session_id()
recursion_limit = 12

final_config = RunnableConfig(
    configurable={"thread_id": session_id},
    recursion_limit=recursion_limit
)

# Agent Execution

This section demonstrates how to interact with the compiled AI agent by providing a user question and streaming the output.

In [ ]:
from pprint import pprint
user_input = HumanMessage(content="hi my name is Bob")
inputs = {"question": user_input}

for output in app.stream(inputs, final_config):
    print(config)
    for key, value in output.items():
        pprint(f"Finished running: {key}:")
    
print("------------------------------************--------------------------------")
print(f"Assistant: {value['generation'].strip()}")
print("------------------------------************--------------------------------\n")

# Simple Agentic Application
This is a simple agentic AI application. When run, it'll generate a new session UUID and you can keep interacting with the AI until "/exit" is entered

Try these questions:
- My name is Bob, I'm 54 and have pink hair
- What's my name?
- How old am I?
- What color hair do I have?

In [ ]:
from pprint import pprint

session_id, config = load_session_id()
recursion_limit = 12

final_config = RunnableConfig(
    configurable={"thread_id": session_id},
    recursion_limit=recursion_limit
)

# Interactive chat loop
print("Chat started! Type '/exit' to quit.\n")

while True:
    user_input_text = input("You: ").strip()
    if user_input_text.lower() == "/exit":
        print("Goodbye!")
        break
        
    if not user_input_text:
        continue
    
    user_input = HumanMessage(content=user_input_text)
    inputs = {"question": user_input}
    
    for output in app.stream(inputs, final_config):
        for key, value in output.items():
            pprint(f"Finished running: {key}:")
    
    # Print the final generation
    print("------------------------------************--------------------------------")
    print(f"Assistant: {value['generation'].strip()}")
    print("------------------------------************--------------------------------\n")